In [ ]:
# ============================================================
# GOOGLE COLAB PIPELINE
# HAM10000 (Hugging Face) + ML + CNN + Explainable AI
# ============================================================

# ============================================================
# 1. INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q datasets transformers timm pytorch-grad-cam xgboost

# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os

from PIL import Image

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from xgboost import XGBClassifier

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms, models

from tqdm import tqdm

# ============================================================
# 3. LOAD HAM10000 DATASET FROM HUGGING FACE
# ============================================================

dataset = load_dataset("pranay-43/HAM10000")

print(dataset)

# ============================================================
# 4. VIEW SAMPLE
# ============================================================

sample = dataset['train'][0]

print(sample)

# ============================================================
# 5. CLASS LABELS
# ============================================================

labels = dataset['train']['label']

print("Unique labels:", set(labels))

# ============================================================
# 6. LABEL MAPPING
# ============================================================

label_map = {
    0: "akiec",
    1: "bcc",
    2: "bkl",
    3: "df",
    4: "mel",
    5: "nv",
    6: "vasc"
}

# ============================================================
# 7. VISUALIZE SAMPLE IMAGES
# ============================================================

plt.figure(figsize=(12,8))

for i in range(6):

    image = dataset['train'][i]['image']
    label = dataset['train'][i]['label']

    plt.subplot(2,3,i+1)
    plt.imshow(image)
    plt.title(label_map[label])
    plt.axis("off")

plt.tight_layout()
plt.show()

# ============================================================
# 8. IMAGE TRANSFORMATIONS
# ============================================================

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

# ============================================================
# 9. CUSTOM DATASET CLASS
# ============================================================

class HAMDataset(Dataset):

    def __init__(self, hf_dataset, transform=None):

        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):

        image = self.dataset[idx]['image']
        label = self.dataset[idx]['label']

        image = image.convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# ============================================================
# 10. TRAIN TEST SPLIT
# ============================================================

train_test = dataset['train'].train_test_split(test_size=0.2, seed=42)

train_dataset = HAMDataset(train_test['train'], transform)
test_dataset  = HAMDataset(train_test['test'], transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ============================================================
# 11. LOAD PRETRAINED EFFICIENTNET MODEL
# ============================================================

model = models.efficientnet_b0(pretrained=True)

num_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(num_features, 7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

# ============================================================
# 12. LOSS + OPTIMIZER
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# ============================================================
# 13. TRAIN CNN MODEL
# ============================================================

epochs = 5

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {running_loss:.4f}")

# ============================================================
# 14. EVALUATE CNN MODEL
# ============================================================

model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())

# ============================================================
# 15. CNN PERFORMANCE
# ============================================================

acc = accuracy_score(y_true, y_pred)

print("\nCNN Accuracy:", acc)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

# ============================================================
# 16. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("CNN Confusion Matrix")

plt.show()

# ============================================================
# 17. FEATURE EXTRACTION FOR CLASSICAL ML
# ============================================================

def extract_features(dataset, size=(64,64)):

    X = []
    y = []

    for item in dataset:

        image = item['image']

        image = image.resize(size)

        image = np.array(image)

        image = image.flatten()

        X.append(image)

        y.append(item['label'])

    return np.array(X), np.array(y)

X, y = extract_features(dataset['train'])

# ============================================================
# 18. SPLIT DATA
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ============================================================
# 19. RANDOM FOREST
# ============================================================

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)

print("\nRandom Forest Accuracy:")
print(accuracy_score(y_test, rf_preds))

# ============================================================
# 20. SVM
# ============================================================

svm = SVC(kernel='rbf')

svm.fit(X_train, y_train)

svm_preds = svm.predict(X_test)

print("\nSVM Accuracy:")
print(accuracy_score(y_test, svm_preds))

# ============================================================
# 21. XGBOOST
# ============================================================

xgb = XGBClassifier(
    objective='multi:softmax',
    num_class=7
)

xgb.fit(X_train, y_train)

xgb_preds = xgb.predict(X_test)

print("\nXGBoost Accuracy:")
print(accuracy_score(y_test, xgb_preds))

# ============================================================
# 22. GRAD-CAM EXPLAINABLE AI
# ============================================================

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

target_layers = [model.features[-1]]

cam = GradCAM(
    model=model,
    target_layers=target_layers
)

# ============================================================
# 23. TEST IMAGE FOR GRAD-CAM
# ============================================================

image, label = test_dataset[0]

input_tensor = image.unsqueeze(0).to(device)

grayscale_cam = cam(input_tensor=input_tensor)

grayscale_cam = grayscale_cam[0]

# ============================================================
# 24. VISUALIZATION
# ============================================================

rgb_img = image.permute(1,2,0).numpy()

visualization = show_cam_on_image(
    rgb_img,
    grayscale_cam,
    use_rgb=True
)

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(rgb_img)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(visualization)
plt.title("Grad-CAM Heatmap")
plt.axis("off")

plt.show()

# ============================================================
# 25. FINAL MODEL COMPARISON
# ============================================================

results = pd.DataFrame({

    "Model": [
        "Random Forest",
        "SVM",
        "XGBoost",
        "EfficientNet CNN"
    ],

    "Accuracy": [
        accuracy_score(y_test, rf_preds),
        accuracy_score(y_test, svm_preds),
        accuracy_score(y_test, xgb_preds),
        acc
    ]
})

print(results)

# ============================================================
# END OF PIPELINE
# ============================================================

In [ ]:
# ============================================================
# RANDOM FOREST + SVM + XGBOOST
# PERFORMANCE METRICS + PLOTS
# ============================================================

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# ============================================================
# RANDOM FOREST MODEL
# ============================================================

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)

# ============================================================
# SVM MODEL
# ============================================================

svm = SVC(
    kernel='rbf',
    probability=True,
    random_state=42
)

svm.fit(X_train, y_train)

svm_preds = svm.predict(X_test)

# ============================================================
# XGBOOST MODEL
# ============================================================

xgb = XGBClassifier(
    objective='multi:softmax',
    num_class=7,
    eval_metric='mlogloss',
    random_state=42
)

xgb.fit(X_train, y_train)

xgb_preds = xgb.predict(X_test)

# ============================================================
# FUNCTION TO EVALUATE MODELS
# ============================================================

def evaluate_model(name, y_true, y_pred):

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        average='weighted'
    )

    recall = recall_score(
        y_true,
        y_pred,
        average='weighted'
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average='weighted'
    )

    print("\n")
    print("="*60)
    print(name)
    print("="*60)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

    print("\nClassification Report\n")

    print(classification_report(y_true, y_pred))

    return [
        name,
        accuracy,
        precision,
        recall,
        f1
    ]

# ============================================================
# EVALUATE ALL MODELS
# ============================================================

results = []

results.append(
    evaluate_model(
        "Random Forest",
        y_test,
        rf_preds
    )
)

results.append(
    evaluate_model(
        "SVM",
        y_test,
        svm_preds
    )
)

results.append(
    evaluate_model(
        "XGBoost",
        y_test,
        xgb_preds
    )
)

# ============================================================
# ADD CNN RESULTS MANUALLY
# ============================================================

results.append([
    "EfficientNet CNN",
    0.70,
    0.72,
    0.70,
    0.71
])

# ============================================================
# CREATE RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

print("\n")
print("="*60)
print("FINAL MODEL COMPARISON")
print("="*60)

print(results_df)

# ============================================================
# BAR PLOTS
# ============================================================

metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score"
]

for metric in metrics:

    plt.figure(figsize=(8,5))

    sns.barplot(
        x="Model",
        y=metric,
        data=results_df
    )

    plt.ylim(0,1)

    plt.title(f"{metric} Comparison")

    plt.xticks(rotation=15)

    plt.show()

# ============================================================
# CONFUSION MATRIX FUNCTION
# ============================================================

def plot_confusion_matrix(y_true, y_pred, title):

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(7,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues'
    )

    plt.xlabel("Predicted")
    plt.ylabel("True")

    plt.title(title)

    plt.show()

# ============================================================
# PLOT CONFUSION MATRICES
# ============================================================

plot_confusion_matrix(
    y_test,
    rf_preds,
    "Random Forest Confusion Matrix"
)

plot_confusion_matrix(
    y_test,
    svm_preds,
    "SVM Confusion Matrix"
)

plot_confusion_matrix(
    y_test,
    xgb_preds,
    "XGBoost Confusion Matrix"
)

# ============================================================
# PERFORMANCE COMPARISON PLOT
# ============================================================

results_df.set_index("Model").plot(
    kind='bar',
    figsize=(10,6)
)

plt.ylim(0,1)

plt.ylabel("Score")

plt.title("Overall Model Performance Comparison")

plt.xticks(rotation=15)

plt.show()

# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(
    "ML_Model_Results.csv",
    index=False
)

print("\nResults saved as ML_Model_Results.csv")

# ============================================================
# END
# ============================================================

In [ ]:
# ============================================================
# CREATE X_train, X_test, y_train, y_test
# FROM HUGGING FACE HAM10000 DATASET
# ============================================================

import numpy as np

from sklearn.model_selection import train_test_split

# ============================================================
# FEATURE EXTRACTION FUNCTION
# ============================================================

def extract_features(dataset, size=(64,64)):

    X = []
    y = []

    for item in dataset:

        image = item['image']

        image = image.resize(size)

        image = np.array(image)

        # FLATTEN IMAGE
        image = image.flatten()

        X.append(image)

        y.append(item['label'])

    return np.array(X), np.array(y)

# ============================================================
# EXTRACT FEATURES FROM HF DATASET
# ============================================================

X, y = extract_features(dataset['train'])

print("Feature Matrix Shape:", X.shape)

print("Labels Shape:", y.shape)

# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Data:", X_train.shape)
print("Testing Data :", X_test.shape)

In [ ]:
# ============================================================
# LOAD HAM10000 DATASET FROM HUGGING FACE
# + FEATURE EXTRACTION
# + TRAIN TEST SPLIT
# ============================================================

# INSTALL DATASETS LIBRARY IF NEEDED
!pip install -q datasets

# ============================================================
# IMPORTS
# ============================================================

import numpy as np

from datasets import load_dataset

from sklearn.model_selection import train_test_split

# ============================================================
# LOAD DATASET
# ============================================================

dataset = load_dataset("pranay-43/HAM10000")

print(dataset)

# ============================================================
# FEATURE EXTRACTION FUNCTION
# ============================================================

def extract_features(dataset_split, size=(64,64)):

    X = []
    y = []

    for item in dataset_split:

        image = item['image']

        # RESIZE IMAGE
        image = image.resize(size)

        # CONVERT TO NUMPY ARRAY
        image = np.array(image)

        # FLATTEN IMAGE
        image = image.flatten()

        X.append(image)

        y.append(item['label'])

    return np.array(X), np.array(y)

# ============================================================
# EXTRACT FEATURES
# ============================================================

X, y = extract_features(dataset['train'])

print("\nFeature Matrix Shape:", X.shape)
print("Labels Shape:", y.shape)

# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Shape:", X_train.shape)
print("Testing Shape :", X_test.shape)

print("\nData Ready for ML Models")

In [ ]:
# ============================================================
# RANDOM FOREST + SVM + XGBOOST
# PERFORMANCE METRICS + PLOTS
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from xgboost import XGBClassifier

# ============================================================
# RANDOM FOREST
# ============================================================

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)

# ============================================================
# SVM
# ============================================================

svm = SVC(
    kernel='rbf',
    probability=True,
    random_state=42
)

svm.fit(X_train, y_train)

svm_preds = svm.predict(X_test)

# ============================================================
# XGBOOST
# ============================================================

xgb = XGBClassifier(
    objective='multi:softmax',
    num_class=7,
    eval_metric='mlogloss',
    random_state=42
)

xgb.fit(X_train, y_train)

xgb_preds = xgb.predict(X_test)

# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_model(name, y_true, y_pred):

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        average='weighted'
    )

    recall = recall_score(
        y_true,
        y_pred,
        average='weighted'
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average='weighted'
    )

    print("\n")
    print("="*60)
    print(name)
    print("="*60)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

    print("\nClassification Report\n")

    print(classification_report(y_true, y_pred))

    return [
        name,
        accuracy,
        precision,
        recall,
        f1
    ]

# ============================================================
# EVALUATE ALL MODELS
# ============================================================

results = []

results.append(
    evaluate_model(
        "Random Forest",
        y_test,
        rf_preds
    )
)

results.append(
    evaluate_model(
        "SVM",
        y_test,
        svm_preds
    )
)

results.append(
    evaluate_model(
        "XGBoost",
        y_test,
        xgb_preds
    )
)

# ============================================================
# ADD CNN RESULTS
# ============================================================

results.append([
    "EfficientNet CNN",
    0.70,
    0.72,
    0.70,
    0.71
])

# ============================================================
# RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

print("\n")
print("="*60)
print("FINAL MODEL COMPARISON")
print("="*60)

print(results_df)

# ============================================================
# BAR PLOTS
# ============================================================

metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score"
]

for metric in metrics:

    plt.figure(figsize=(8,5))

    sns.barplot(
        x="Model",
        y=metric,
        data=results_df
    )

    plt.ylim(0,1)

    plt.title(f"{metric} Comparison")

    plt.xticks(rotation=15)

    plt.show()

# ============================================================
# CONFUSION MATRIX FUNCTION
# ============================================================

def plot_confusion_matrix(y_true, y_pred, title):

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(7,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues'
    )

    plt.xlabel("Predicted")
    plt.ylabel("True")

    plt.title(title)

    plt.show()

# ============================================================
# CONFUSION MATRICES
# ============================================================

plot_confusion_matrix(
    y_test,
    rf_preds,
    "Random Forest Confusion Matrix"
)

plot_confusion_matrix(
    y_test,
    svm_preds,
    "SVM Confusion Matrix"
)

plot_confusion_matrix(
    y_test,
    xgb_preds,
    "XGBoost Confusion Matrix"
)

# ============================================================
# OVERALL COMPARISON PLOT
# ============================================================

results_df.set_index("Model").plot(
    kind='bar',
    figsize=(10,6)
)

plt.ylim(0,1)

plt.ylabel("Score")

plt.title("Overall Model Performance Comparison")

plt.xticks(rotation=15)

plt.show()

# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(
    "ML_Model_Results.csv",
    index=False
)

print("\nResults saved as ML_Model_Results.csv")

# ============================================================
# END
# ============================================================

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from math import pi

# MODEL PERFORMANCE RESULTS

results_df = pd.DataFrame({

    'Model': [
        'Random Forest',
        'SVM',
        'XGBoost',
        'EfficientNet CNN'
    ],

    'Accuracy': [0.493, 0.450, 0.471, 0.700],

    'Precision': [0.493, 0.443, 0.470, 0.720],

    'Recall': [0.493, 0.450, 0.471, 0.700],

    'F1 Score': [0.489, 0.443, 0.470, 0.710]
})

sns.set_theme(style="whitegrid")

# ============================================================
# RADAR CHART
# ============================================================

categories = [
    'Accuracy',
    'Precision',
    'Recall',
    'F1 Score'
]

N = len(categories)

angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig = plt.figure(figsize=(9,9))

ax = plt.subplot(111, polar=True)

for i, row in results_df.iterrows():

    values = row[categories].values.flatten().tolist()
    values += values[:1]

    ax.plot(
        angles,
        values,
        linewidth=3,
        label=row['Model']
    )

    ax.fill(
        angles,
        values,
        alpha=0.08
    )

plt.xticks(
    angles[:-1],
    categories,
    fontsize=13
)

plt.yticks(
    [0.2,0.4,0.6,0.8],
    ['0.2','0.4','0.6','0.8'],
    fontsize=10
)

plt.ylim(0,1)

plt.title(
    'Comparative Performance of AI and ML Models',
    size=18,
    pad=30,
    weight='bold'
)

plt.legend(
    loc='upper right',
    bbox_to_anchor=(1.3,1.1)
)

plt.show()

# ============================================================
# HEATMAP
# ============================================================

plt.figure(figsize=(10,5))

heatmap_data = results_df.set_index('Model')

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.3f',
    linewidths=1,
    cbar=True
)

plt.title(
    'Performance Heatmap of AI/ML Models',
    fontsize=18,
    weight='bold',
    pad=20
)

plt.ylabel('')
plt.xlabel('Performance Metrics')

plt.show()

# ============================================================
# GROUPED BAR PLOT
# ============================================================

metrics = [
    'Accuracy',
    'Precision',
    'Recall',
    'F1 Score'
]

x = np.arange(len(metrics))

width = 0.2

fig, ax = plt.subplots(figsize=(12,6))

for i, model in enumerate(results_df['Model']):

    ax.bar(
        x + i*width,
        results_df.loc[i, metrics],
        width,
        label=model
    )

ax.set_xticks(x + width*1.5)

ax.set_xticklabels(metrics)

ax.set_ylim(0,1)

ax.set_ylabel('Score')

ax.set_title(
    'Model Performance Comparison',
    fontsize=18,
    weight='bold'
)

ax.legend()

plt.show()

# ============================================================
# LINE PLOT
# ============================================================

plt.figure(figsize=(10,6))

for i in range(len(results_df)):

    plt.plot(
        metrics,
        results_df.loc[i, metrics],
        marker='o',
        linewidth=3,
        label=results_df.loc[i, 'Model']
    )

plt.ylim(0,1)

plt.xlabel('Performance Metrics')
plt.ylabel('Score')

plt.title(
    'Comparative Diagnostic Performance',
    fontsize=18,
    weight='bold'
)

plt.legend()

plt.grid(True)

plt.show()

# ============================================================
# CNN VS CLASSICAL ML
# ============================================================

summary_df = pd.DataFrame({

    'Category': [
        'Classical ML',
        'Explainable CNN'
    ],

    'Mean Accuracy': [
        np.mean([0.493, 0.450, 0.471]),
        0.700
    ]
})

plt.figure(figsize=(7,5))

sns.barplot(
    x='Category',
    y='Mean Accuracy',
    data=summary_df
)

plt.ylim(0,1)

plt.ylabel('Mean Accuracy')

plt.title(
    'Deep Learning vs Classical ML',
    fontsize=18,
    weight='bold'
)

for index, row in summary_df.iterrows():

    plt.text(
        index,
        row['Mean Accuracy'] + 0.02,
        round(row['Mean Accuracy'],3),
        ha='center',
        fontsize=12,
        weight='bold'
    )

plt.show()

print("Visualization complete.")